In [ ]:
!pip install ortools pandas numpy -q

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import pandas as pd
import numpy as np
from datetime import datetime

print("OR-Tools installed")

class ZoneModelSolver:
    """
    Zone Model: Vehicle-customer assignment with geographic restrictions
    Vehicles restricted to specific distance zones from depot
    """

    def __init__(self, processed_data):
        self.data = processed_data
        self.depot = processed_data['depot']
        self.customers_home = processed_data['customers_home']
        self.customers_locker = processed_data['customers_locker']
        self.lockers = processed_data['lockers']
        self.vehicles = processed_data['vehicles']
        self.distance_dict = processed_data['distance_dict']
        self.locker_assignments = processed_data['locker_assignments']
        self.params = processed_data['params']
        self.zones = processed_data['zones']

    def solve(self):
        print("\n" + "=" * 60)
        print("ZONE MODEL - GEOGRAPHIC RESTRICTIONS")
        print("=" * 60)

        start_time = datetime.now()

        print("\n Step 1: Node list")

        locations = [self.depot['id']]
        home_ids = self.customers_home['Customer_ID'].tolist()
        used_lockers = list(set(self.locker_assignments.values()))

        locations.extend(home_ids)
        locations.extend(used_lockers)

        print(f"  Total nodes: {len(locations)}")
        print(f"    - Depot: 1")
        print(f"    - Home customers: {len(home_ids)}")
        print(f"    - Used lockers: {len(used_lockers)}")

        print("\n Step 2: Distances from depot & assigning zones")

        node_distances = {}
        node_zones = {}

        for loc in locations:
            if loc == self.depot['id']:
                node_distances[loc] = 0
                node_zones[loc] = 'All'
            else:
                dist = self.get_distance(self.depot['id'], loc)
                node_distances[loc] = dist

                if dist <= self.zones['Bicycle']['max']:
                    node_zones[loc] = 'Bicycle'
                elif dist <= self.zones['Electric']['max']:
                    node_zones[loc] = 'Electric'
                else:
                    node_zones[loc] = 'Diesel'

        zone_counts = {'Bicycle': 0, 'Electric': 0, 'Diesel': 0}
        for loc, zone in node_zones.items():
            if zone != 'All':
                zone_counts[zone] += 1

        print(f"\n  Zone distribution:")
        print(f"    - Bicycle zone (0-2.5 km): {zone_counts['Bicycle']} nodes")
        print(f"    - Electric zone (2.5-7 km): {zone_counts['Electric']} nodes")
        print(f"    - Diesel zone (7+ km): {zone_counts['Diesel']} nodes")

        print("\n Step 3: Distance matrix")

        distance_matrix = []
        for from_loc in locations:
            row = []
            for to_loc in locations:
                if from_loc == to_loc:
                    row.append(0)
                else:
                    dist = self.get_distance(from_loc, to_loc)
                    row.append(int(dist * 100000))
            distance_matrix.append(row)

        print(f" Matrix size: {len(distance_matrix)}x{len(distance_matrix[0])}")

        print("\n Step 4: Demand arrays")

        demands = [0]

        for _ in home_ids:
            demands.append(1)

        for locker_id in used_lockers:
            num_customers = sum(1 for cust_id, assigned_locker
                              in self.locker_assignments.items()
                              if assigned_locker == locker_id)
            demands.append(num_customers)

        print(f"  Total demand: {sum(demands)} parcels")
        print(f"    - Home deliveries: {len(home_ids)}")
        print(f"    - Locker deliveries: {sum(demands[len(home_ids)+1:])}")

        print("\n Step 5: Routing model")

        manager = pywrapcp.RoutingIndexManager(
            len(locations),
            len(self.vehicles),
            0
        )

        routing = pywrapcp.RoutingModel(manager)

        print(f"  Model created: {len(locations)} nodes, {len(self.vehicles)} vehicles")

        print("\n Step 6: Registering callbacks")

        def distance_callback(from_index, to_index):
            from_node = manager.IndexToNode(from_index)
            to_node = manager.IndexToNode(to_index)
            return distance_matrix[from_node][to_node]

        distance_callback_index = routing.RegisterTransitCallback(distance_callback)

        for vehicle_id in range(len(self.vehicles)):
            v_row = self.vehicles.iloc[vehicle_id]
            emission_factor = float(v_row['Emission_gCO2_per_km'])

            def make_emission_callback(vid, ef):
                def callback(from_index, to_index):
                    from_node = manager.IndexToNode(from_index)
                    to_node = manager.IndexToNode(to_index)
                    dist_km = distance_matrix[from_node][to_node] / 100000.0
                    return int(dist_km * ef * 100)
                return callback

            emission_callback = make_emission_callback(vehicle_id, emission_factor)
            emission_callback_index = routing.RegisterTransitCallback(emission_callback)
            routing.SetArcCostEvaluatorOfVehicle(emission_callback_index, vehicle_id)

        print(f"  Emissions callbacks registered for {len(self.vehicles)} vehicles")

        print("\n Step 7: Vehicle capacity constraints")

        def demand_callback(from_index):
            from_node = manager.IndexToNode(from_index)
            return demands[from_node]

        demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
        vehicle_capacities = [int(v['Parcel_Capacity']) for _, v in self.vehicles.iterrows()]

        routing.AddDimensionWithVehicleCapacity(
            demand_callback_index,
            0,
            vehicle_capacities,
            True,
            'Capacity'
        )

        print(f"  Capacity constraints: {vehicle_capacities}")

        print("\n Step 8: Range constraints")

        routing.AddDimension(
            distance_callback_index,
            0,
            300000000,
            True,
            'Distance'
        )

        distance_dimension = routing.GetDimensionOrDie('Distance')

        for vehicle_id in range(len(self.vehicles)):
            v_row = self.vehicles.iloc[vehicle_id]
            max_range_km = float(v_row['Range_km'])
            max_range_cm = int(max_range_km * 100000)

            index = routing.End(vehicle_id)
            distance_dimension.CumulVar(index).SetMax(max_range_cm)

        print(f"  Range constraints set for {len(self.vehicles)} vehicles")

        print("\n Step 9: Time Constraints")

        s_home_base = self.params['service_time_home_base']
        s_home_per = self.params['service_time_home_per']
        s_locker_base = self.params['service_time_locker_base']
        s_locker_per = self.params['service_time_locker_per']
        max_shift_minutes = self.params['max_shift_minutes']

        for vehicle_id in range(len(self.vehicles)):
            v_row = self.vehicles.iloc[vehicle_id]
            speed_kmh = float(v_row['Speed_kmh'])

            def make_time_callback(vid, spd, s_hb, s_hp, s_lb, s_lp, homes, lockers, dems, locs):
                def callback(from_index, to_index):
                    from_node = manager.IndexToNode(from_index)
                    to_node = manager.IndexToNode(to_index)

                    dist_km = distance_matrix[from_node][to_node] / 100000.0
                    travel_time_min = (dist_km / spd) * 60

                    service_time = 0
                    if to_node > 0:
                        to_location = locs[to_node]
                        parcel_count = dems[to_node]

                        if to_location in homes:
                            service_time = s_hb + (parcel_count * s_hp)
                        elif to_location in lockers:
                            service_time = s_lb + (parcel_count * s_lp)

                    total_time_seconds = int((travel_time_min + service_time) * 60)
                    return total_time_seconds

                return callback

            time_callback = make_time_callback(
                vehicle_id, speed_kmh,
                s_home_base, s_home_per,
                s_locker_base, s_locker_per,
                home_ids, used_lockers, demands, locations
            )

            time_callback_index = routing.RegisterTransitCallback(time_callback)

            routing.AddDimension(
                time_callback_index,
                0,
                int(max_shift_minutes * 60),
                True,
                f'Time_{vehicle_id}'
            )

        print(f"  Time constraints with service time added")

        print("\n Step 10: Zone restrictions")

        zone_violations_prevented = 0

        for vehicle_id in range(len(self.vehicles)):
            v_row = self.vehicles.iloc[vehicle_id]
            vehicle_type = v_row['Vehicle_Type']

            print(f"\n  Setting restrictions for {v_row['Vehicle_ID']} ({vehicle_type}):")

            for node_idx in range(1, len(locations)):
                node_id = locations[node_idx]
                node_zone = node_zones[node_id]

                if vehicle_type != node_zone:
                    index = manager.NodeToIndex(node_idx)
                    routing.VehicleVar(index).RemoveValue(vehicle_id)
                    zone_violations_prevented += 1

            allowed = sum(1 for nz in node_zones.values() if nz == vehicle_type or nz == 'All')
            print(f"   Can visit: {allowed} /{len(locations)} nodes")

        print(f"\n  Total zone restrictions applied: {zone_violations_prevented}")

        print("\n Step 11: Search parameters")

        search_parameters = pywrapcp.DefaultRoutingSearchParameters()
        search_parameters.first_solution_strategy = (
            routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
        )
        search_parameters.local_search_metaheuristic = (
            routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        )
        search_parameters.time_limit.seconds = 600

        print(f"  Method: PATH_CHEAPEST_ARC + Guided Local Search")
        print(f"  Time limit: 600 seconds")

        print("\n Step 10: Solving model")

        solution = routing.SolveWithParameters(search_parameters)

        if not solution:
            print(" NO SOLUTION FOUND!")
            print("\n  Possible reasons:")
            print("  - Zone restrictions too strict")
            print("  - Insufficient vehicles in certain zones")
            print("  - Capacity/time constraints infeasible with zones")
            return None

        solve_time = (datetime.now() - start_time).total_seconds()

        print(f"\n SOLUTION FOUND in {solve_time:.1f} seconds")

        result = self._extract_solution(
            manager, routing, solution, locations,
            home_ids, used_lockers, distance_matrix, demands, node_zones
        )

        return result

    def _extract_solution(self, manager, routing, solution, locations,
                         home_ids, used_lockers, distance_matrix, demands, node_zones):
        """Extract and format solution"""

        vehicles_data = []
        total_distance = 0
        total_emissions = 0
        total_time = 0
        max_time = 0

        for vehicle_id in range(len(self.vehicles)):
            v_row = self.vehicles.iloc[vehicle_id]

            index = routing.Start(vehicle_id)
            route = []
            route_distance = 0
            route_time = 0
            route_parcels = 0
            homes_visited = 0
            lockers_visited = 0

            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                if node > 0:
                    location = locations[node]
                    route.append(location)
                    route_parcels += demands[node]

                    if location in home_ids:
                        homes_visited += 1
                    elif location in used_lockers:
                        lockers_visited += 1

                previous_index = index
                index = solution.Value(routing.NextVar(index))

                from_node = manager.IndexToNode(previous_index)
                to_node = manager.IndexToNode(index)
                route_distance += distance_matrix[from_node][to_node] / 100000.0

            if len(route) > 0:
                time_dim = routing.GetDimensionOrDie(f'Time_{vehicle_id}')
                end_index = routing.End(vehicle_id)
                route_time = solution.Value(time_dim.CumulVar(end_index)) / 60.0

                route_emissions = route_distance * float(v_row['Emission_gCO2_per_km'])
                capacity = int(v_row['Parcel_Capacity'])
                max_range = float(v_row['Range_km'])

                vehicles_data.append({
                    'Vehicle_ID': v_row['Vehicle_ID'],
                    'Vehicle_Type': v_row['Vehicle_Type'],
                    'Zone_Assigned': v_row['Vehicle_Type'],
                    'Homes_Visited': homes_visited,
                    'Lockers_Visited': lockers_visited,
                    'Total_Stops': len(route),
                    'Parcels_Delivered': route_parcels,
                    'Capacity': capacity,
                    'Utilization_%': round((route_parcels / capacity) * 100, 1),
                    'Distance_km': round(route_distance, 2),
                    'Range_km': max_range,
                    'Range_OK': '(Satisfied)' if route_distance <= max_range else '(Violated)',
                    'Time_min': round(route_time, 1),
                    'Time_formatted': f"{int(route_time//60)}h {int(route_time%60)}m",
                    'Max_Time_min': self.params['max_shift_minutes'],
                    'Time_OK': '(Satisfied)' if route_time <= self.params['max_shift_minutes'] else '(Violated)',
                    'Emissions_g': round(route_emissions, 1),
                    'Route': ' → '.join(route)
                })

                total_distance += route_distance
                total_emissions += route_emissions
                total_time += route_time
                max_time = max(max_time, route_time)

        vehicles_df = pd.DataFrame(vehicles_data)

        locker_data = []
        for customer_id, locker_id in self.locker_assignments.items():
            walking_dist = self.get_distance(customer_id, locker_id)

            locker_data.append({
                'Customer_ID': customer_id,
                'Locker_ID': locker_id,
                'Walking_Distance_km': round(walking_dist, 2)
            })

        lockers_df = pd.DataFrame(locker_data)

        print("\n" + "=" * 60)
        print("SOLUTION - ZONE MODEL")
        print("=" * 60)

        for _, v in vehicles_df.iterrows():
            print(f"\n{v['Vehicle_ID']} ({v['Vehicle_Type']}) - Zone: {v['Zone_Assigned']}")
            print(f"  Stops: {v['Total_Stops']} ({v['Homes_Visited']} homes + {v['Lockers_Visited']} lockers)")
            print(f"  Parcels: {v['Parcels_Delivered']}/{v['Capacity']} ({v['Utilization_%']:.0f}%)")
            print(f"  Distance: {v['Distance_km']} km / {v['Range_km']} km {v['Range_OK']}")
            print(f"  Time: {v['Time_formatted']} ({v['Time_min']} min) / {v['Max_Time_min']} min {v['Time_OK']}")
            print(f"  Emissions: {v['Emissions_g']:.0f} g CO₂")

        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)

        print(f"\n Customers:")
        print(f"  Home: {len(home_ids)}/{len(home_ids)}")
        print(f"  Locker: {len(self.locker_assignments)}/{len(self.locker_assignments)}")
        print(f"  TOTAL: {sum(demands)}/{sum(demands)}")

        avg_time = total_time / len(vehicles_df) if len(vehicles_df) > 0 else 0
        print(f"\n Performance:")
        print(f"  Total distance: {total_distance:.1f} km")
        print(f"  Total time: {int(total_time//60)}h {int(total_time%60)}m ({total_time:.1f} min)")
        print(f"  Average time per vehicle: {int(avg_time//60)}h {int(avg_time%60)}m")
        print(f"  Longest route: {int(max_time//60)}h {int(max_time%60)}m")
        print(f"  Total emissions: {total_emissions:.0f} g CO₂")
        print(f"  Per parcel: {total_emissions/sum(demands):.1f} g/parcel")

        capacity_ok = all(vehicles_df['Utilization_%'] <= 100)
        range_ok = all(vehicles_df['Range_OK'] == '(Satisfied)')
        time_ok = all(vehicles_df['Time_OK'] == '(Satisfied)')

        print(f"\n Vehicles:")
        print(f"  Used: {len(vehicles_df)}/{len(self.vehicles)}")
        print(f"  Capacity: {'All satisfied' if capacity_ok else 'Violated'}")
        print(f"  Range: {'All satisfied' if range_ok else 'Violated'}")
        print(f"  Time: {'All satisfied' if time_ok else 'Violated'}")
        print(f"  Zone restrictions: Enforced")

        all_ok = capacity_ok and range_ok and time_ok
        if all_ok:
            print(f"\n All customers served with zone restrictions satisfied")
        else:
            printprint(f"\n Some constraints violated - check solution")

        print("\n" + "=" * 60)
        print("VEHICLE DETAILS")
        print("=" * 60)

        display_df = vehicles_df[[
            'Vehicle_ID', 'Vehicle_Type', 'Zone_Assigned', 'Homes_Visited', 'Lockers_Visited',
            'Parcels_Delivered', 'Utilization_%', 'Distance_km', 'Range_OK',
            'Time_formatted', 'Time_OK', 'Emissions_g'
        ]].copy()

        display_df.columns = [
            'Vehicle', 'Type', 'Zone', 'Homes', 'Lockers', 'Parcels', 'Util_%',
            'Distance_km', 'Range_OK', 'Time', 'Time_OK', 'Emissions_g'
        ]

        print(f"\n{display_df.to_string(index=False)}")
        print("=" * 60)

        return {
            'vehicles': vehicles_df,
            'lockers': lockers_df,
            'total_distance': total_distance,
            'total_emissions': total_emissions,
            'total_time': total_time,
            'parcels': sum(demands),
            'vehicles_used': len(vehicles_df)
        }

    def get_distance(self, from_id, to_id):
        """Get distance from dictionary"""
        key = (from_id, to_id)
        if key in self.distance_dict:
            return self.distance_dict[key]
        reverse = (to_id, from_id)
        if reverse in self.distance_dict:
            return self.distance_dict[reverse]
        return 0.1

def run_zone_model(processed_data):
    """Run the zone model"""

    solver = ZoneModelSolver(processed_data)
    result = solver.solve()

    if result:
        print("\n ZONE MODEL COMPLETE!")
        return result
    else:
        print("\n ZONE MODEL FAILED!")
        return None

if __name__ == "__main__":
    if 'processed_data' in globals():
        result_zone = run_zone_model(processed_data)
    else:
        print(" Please run Part 1 first to load processed_data")